In [2]:
import time
import pandas as pd
import requests
from pathlib import Path

BASE = "https://mobidata-bw.de/daten/eco-counter/v2/fahrradzaehler_stundenwerten_{}.csv.gz"
CACHE = Path("../../data/raw")
CACHE.mkdir(parents=True, exist_ok=True)

session = requests.Session()
session.headers["User-Agent"] = "fahrrad-analyse (private research project; d.roca.6@proton.me)"

def fetch_month(ym):
    """Download one monthly file unless it is already cached. Returns the path or None."""
    f = CACHE / f"{ym}.csv.gz"
    if f.exists():
        return f                      # cached: no request at all
    for attempt in range(3):
        try:
            r = session.get(BASE.format(ym), timeout=(10, 60))  # (connect, read) seconds
        except requests.RequestException as e:
            print(f"{ym}: {e}")
        else:
            if r.status_code == 404:
                print(f"{ym}: not available")
                return None
            if r.ok:
                f.write_bytes(r.content)
                return f
            print(f"{ym}: HTTP {r.status_code}")
        time.sleep(10 * 2**attempt)   # back off 10 s, 20 s, 40 s on errors
    return None

frames = []
for m in pd.period_range("2021-01", "2026-10", freq="M"):
    ym = m.strftime("%Y%m")
    cached = (CACHE / f"{ym}.csv.gz").exists()
    f = fetch_month(ym)
    if f is not None:
        frames.append(pd.read_csv(f))
    if not cached:
        time.sleep(2)                 # pause between real downloads

hourly = pd.concat(frames, ignore_index=True)

/var/folders/32/vygvwqrx4qx5g6w9q4y9ct180000gn/T/ipykernel_5436/589937503.py:40: DtypeWarning: Columns (14: site_temperature, 15: site_rain_accumulation) have mixed types. Specify dtype option on import or set low_memory=False.
  frames.append(pd.read_csv(f))
/var/folders/32/vygvwqrx4qx5g6w9q4y9ct180000gn/T/ipykernel_5436/589937503.py:40: DtypeWarning: Columns (12: channels_unknown, 14: site_temperature, 15: site_rain_accumulation) have mixed types. Specify dtype option on import or set low_memory=False.
  frames.append(pd.read_csv(f))


In [3]:
ts = pd.to_datetime(hourly["iso_timestamp"], utc=True)
coverage = (hourly.assign(ts=ts)
            .groupby(["domain_name", "counter_site"])["ts"]
            .agg(["min", "max", "count"]))
coverage[coverage["min"] < "2022-01-01"]

min  \
domain_name                counter_site                                                                   
Landeshauptstadt Stuttgart Am Kräherwald                                      2020-12-31 23:00:00+00:00   
                           Böblinger Straße                                   2020-12-31 23:00:00+00:00   
                           Hardware für Waiblinger-/ Taubenheimstr.           2020-12-31 23:00:00+00:00   
                           Inselstraße                                        2020-12-31 23:00:00+00:00   
                           Kirchheimer Straße                                 2020-12-31 23:00:00+00:00   
...                                                                                                 ...   
Stadt Tübingen             Fuß- & Radtunnel Südportal - Derendinger Allee     2020-12-31 23:00:00+00:00   
                           Neckartalradweg Hirschau - parallel L371           2020-12-31 23:00:00+00:00   
                           Unterführung Steinlach/Karlstraße Südseite - St... 2020-12-31 23:00:00+00:00   
Stadt Ulm                  Blautal Lupferbrücke                               2020-12-31 23:00:00+00:00   
Stadtverwaltung Aalen      Radweg am Kocher                                   2020-12-31 23:00:00+00:00   

                                                                                                    max  \
domain_name                counter_site                                                                   
Landeshauptstadt Stuttgart Am Kräherwald                                      2026-10-31 22:00:00+00:00   
                           Böblinger Straße                                   2026-10-31 22:00:00+00:00   
                           Hardware für Waiblinger-/ Taubenheimstr.           2026-10-31 22:00:00+00:00   
                           Inselstraße                                        2026-10-31 22:00:00+00:00   
                           Kirchheimer Straße                                 2026-10-31 22:00:00+00:00   
...                                                                                                 ...   
Stadt Tübingen             Fuß- & Radtunnel Südportal - Derendinger Allee     2026-10-31 22:00:00+00:00   
                           Neckartalradweg Hirschau - parallel L371           2026-10-31 22:00:00+00:00   
                           Unterführung Steinlach/Karlstraße Südseite - St... 2026-10-31 22:00:00+00:00   
Stadt Ulm                  Blautal Lupferbrücke                               2026-10-31 22:00:00+00:00   
Stadtverwaltung Aalen      Radweg am Kocher                                   2026-10-31 22:00:00+00:00   

                                                                               count  
domain_name                counter_site                                               
Landeshauptstadt Stuttgart Am Kräherwald                                       51114  
                           Böblinger Straße                                    50409  
                           Hardware für Waiblinger-/ Taubenheimstr.            50611  
                           Inselstraße                                         51114  
                           Kirchheimer Straße                                  51114  
...                                                                              ...  
Stadt Tübingen             Fuß- & Radtunnel Südportal - Derendinger Allee      51061  
                           Neckartalradweg Hirschau - parallel L371            46431  
                           Unterführung Steinlach/Karlstraße Südseite - St...  45748  
Stadt Ulm                  Blautal Lupferbrücke                                50365  
Stadtverwaltung Aalen      Radweg am Kocher                                    46572  

[96 rows x 3 columns]

In [4]:
coverage = (hourly.assign(ts=ts)
            .groupby(["counter_site_id"])["ts"]
            .agg(["min", "max", "count"]))
coverage = coverage[coverage["min"] < "2022-01-01"]

In [5]:
selected_counter_mask = hourly.loc[:, 'counter_site_id'].isin(coverage.index)
hourly = hourly.loc[selected_counter_mask, :]

In [ ]:
hourly.to_csv("../../data/counters_raw.csv")